In [0]:
ADLS_BASE = "abfss://cryptoblob@cryptoanalysisadls.dfs.core.windows.net"
GOLD_PATH = f"{ADLS_BASE}/gold/"
SILVER_PATH = f"{ADLS_BASE}/silver/"
BRONZE_PATH = f"{ADLS_BASE}/bronze/"

In [0]:
from delta.tables import DeltaTable
import pyspark.sql.functions as F
from pyspark.sql.window import Window

gold_df = spark.read.format("delta").load(GOLD_PATH) if DeltaTable.isDeltaTable(spark, GOLD_PATH) else None

def upsert_gold_latest(df, gold_path):

    window_spec = Window.partitionBy("coin_id").orderBy(F.col("timestamp").desc())

    df_latest_batch = (
        df.withColumn("rn", F.row_number().over(window_spec))
          .filter(F.col("rn") == 1)
          .drop("rn")
    )

    if not DeltaTable.isDeltaTable(spark, gold_path):
        (
            df_latest_batch.write
                .format("delta")
                .mode("overwrite")
                .save(gold_path)
        )
        return

    delta_tbl = DeltaTable.forPath(spark, gold_path)

    (
        delta_tbl.alias("t")
        .merge(
            df_latest_batch.alias("s"),
            "t.coin_id = s.coin_id"
        )
        .whenMatchedUpdate(
            condition="s.timestamp > t.timestamp",
            set={col: f"s.{col}" for col in df_latest_batch.columns}
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
silver_df = spark.read.format("delta").load(SILVER_PATH)

upsert_gold_latest(silver_df, f"{GOLD_PATH}/latest_snapshot/")

In [0]:
df_latest = spark.read.format("delta").load(f"{GOLD_PATH}/latest_snapshot/")

In [0]:
# Aggregated Tables — Top 10 Gainers / Losers / Volume


import pyspark.sql.functions as F

from delta.tables import DeltaTable

def write_gold_table(df, path: str):

    # If path exists but is NOT delta → delete it
    if not DeltaTable.isDeltaTable(spark, path):
        try:
            dbutils.fs.rm(path, True)
        except:
            pass

    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(path)


# Top 20 Gainers 
df_top_gainers = (
    df_latest
    .filter(F.col("price_change").isNotNull())
    .orderBy(F.col("price_change").desc())
    .limit(20)
)

write_gold_table(df_top_gainers, f"{GOLD_PATH}/top_gainers/")


# Top 20 Losers 
df_top_losers = (
    df_latest
    .filter(F.col("price_change").isNotNull())
    .orderBy(F.col("price_change").asc())
    .limit(20)
)

write_gold_table(df_top_losers, f"{GOLD_PATH}/top_losers/")


# Top 20 by Volume 
df_top_volume = (
    df_latest
    .filter(F.col("total_volume").isNotNull())
    .orderBy(F.col("total_volume").desc())
    .limit(20)
)

write_gold_table(df_top_volume, f"{GOLD_PATH}/top_volume/")

In [0]:
from delta.tables import DeltaTable
import pyspark.sql.functions as F
from pyspark.sql.window import Window

def upsert_dim_coin(df, path):

    df_dim = (
        df.select("coin_id", "symbol", "name")
          .dropDuplicates(["coin_id"])
          .withColumn("effective_from", F.current_timestamp())
          .withColumn("effective_to", F.lit(None).cast("timestamp"))
          .withColumn("is_current", F.lit(True))
    )

    # FIRST LOAD
    if not DeltaTable.isDeltaTable(spark, path):
        df_dim = df_dim.withColumn(
            "coin_sk",
            F.row_number().over(Window.orderBy("coin_id"))
        )

        df_dim.write.format("delta").mode("overwrite").save(path)
        return

    delta_tbl = DeltaTable.forPath(spark, path)
    tgt_df = delta_tbl.toDF()


    # DETECT CHANGES
    updates = (
        df_dim.alias("s")
        .join(tgt_df.alias("t"), "coin_id")
        .where(
            (F.col("t.is_current") == True) &
            (
                (F.col("s.name") != F.col("t.name")) |
                (F.col("s.symbol") != F.col("t.symbol"))
            )
        )
        .select("s.*")
    )

    # EXPIRE OLD RECORDS
 
    (
        delta_tbl.alias("t")
        .merge(
            updates.alias("s"),
            "t.coin_id = s.coin_id AND t.is_current = true"
        )
        .whenMatchedUpdate(set={
            "is_current": "false",
            "effective_to": "s.effective_from"
        })
        .execute()
    )

    # GET MAX SK (for deterministic keys)

    max_sk = tgt_df.agg(F.max("coin_sk")).collect()[0][0]
    max_sk = max_sk if max_sk is not None else 0


    # NEW + UPDATED ROWS TO INSERT

    inserts = (
        df_dim.alias("s")
        .join(
            tgt_df.alias("t"),
            (F.col("s.coin_id") == F.col("t.coin_id")) & (F.col("t.is_current") == True),
            "left_anti"
        )
    )

    # assign deterministic surrogate keys
    inserts = inserts.withColumn(
        "coin_sk",
        F.row_number().over(Window.orderBy("coin_id")) + max_sk
    )


    # APPEND NEW RECORDS

    inserts.select(
        "coin_sk",
        "coin_id",
        "symbol",
        "name",
        "effective_from",
        "effective_to",
        "is_current"
    ).write.format("delta").mode("append").save(path)

In [0]:
from pyspark.sql.functions import col, year, month, dayofmonth, date_format, weekofyear
from pyspark.sql.functions import monotonically_increasing_id

def build_dim_date(df, path):

    df_date = (
        df.select(F.to_date("ingestion_time").alias("date"))
          .dropDuplicates()
          .withColumn("year", year("date"))
          .withColumn("month", month("date"))
          .withColumn("day", dayofmonth("date"))
          .withColumn("day_of_week", date_format("date", "EEEE"))
          .withColumn("week_of_year", weekofyear("date"))
          .withColumn("date_sk", monotonically_increasing_id())
    )

    if not DeltaTable.isDeltaTable(spark, path):
        df_date.write.format("delta").mode("overwrite").save(path)
    else:
        existing = spark.read.format("delta").load(path)

        new_dates = df_date.join(existing, "date", "left_anti")

        new_dates.write.format("delta").mode("append").save(path)


build_dim_date(silver_df, f"{GOLD_PATH}/data_modelling/dim_date/")

In [0]:

# Step 5: Fact Table — crypto_price_fact (PROPER DW DESIGN)


from pyspark.sql.functions import col, to_date, monotonically_increasing_id
from delta.tables import DeltaTable

# Load dimensions
dim_coin = spark.read.format("delta").load(f"{GOLD_PATH}/data_modelling/dim_coin/")   # has coin_sk, coin_id
dim_date = spark.read.format("delta").load(f"{GOLD_PATH}/data_modelling/dim_date/")   # has date_sk, date


# Join with dimensions to get surrogate keys

df_fact = (
    silver_df
    .join(dim_coin, silver_df["coin_id"] == dim_coin["coin_id"], "left")
    .withColumn("date", to_date(col("ingestion_time")))
    .join(dim_date, "date", "left")
)

df_fact = df_fact.select(
    monotonically_increasing_id().alias("fact_id"),

    col("coin_sk"),
    col("date_sk"),

    col("ingestion_time"),   # keep only once

    col("current_price").alias("price"),
    col("market_cap"),
    col("total_volume").alias("volume"),
    col("price_change").alias("price_change_pct"),
    col("high_24h"),
    col("low_24h"),

    col("volatility_flag"),
    col("volume_spike_flag"),
    col("is_anomaly")
)

(
    df_fact.write
           .format("delta")
           .mode("append")  
           .partitionBy("date_sk")  # better than raw date
           .save(f"{GOLD_PATH}/data_modelling/fact_table/")
)

In [0]:

# Anomaly Detection (Aligned with pipeline)


import pyspark.sql.functions as F

def detect_anomalies(df):
    """Identifies anomalous crypto behavior."""

    # Compute average volume (batch-level)
    avg_vol = df.agg(F.avg(F.col("total_volume")).alias("avg_vol")).collect()[0]["avg_vol"]
    avg_vol = avg_vol if avg_vol else 0.0

    df_anomaly = (
        df
        .filter(
            (F.abs(F.col("price_change")) > 15) |
            (F.col("total_volume") > (2 * avg_vol))
        )
        .withColumn(
            "anomaly_reason",
            F.when(
                (F.abs(F.col("price_change")) > 15) &
                (F.col("total_volume") > (2 * avg_vol)),
                F.lit("PRICE_SPIKE + VOLUME_SPIKE")
            ).when(
                F.abs(F.col("price_change")) > 15,
                F.lit("PRICE_SPIKE")
            ).otherwise(
                F.lit("VOLUME_SPIKE")
            )
        )
        .select(
            "coin_id", "name", "symbol",
            "current_price",
            "price_change",
            "total_volume",
            "anomaly_reason",
            "ingestion_time"
        )
        .orderBy(F.abs(F.col("price_change")).desc())
    )

    return df_anomaly


# Gold latest is your source
df_latest = spark.read.format("delta").load(f"{GOLD_PATH}/latest_snapshot/")

df_anomalies = detect_anomalies(df_latest)

(
    df_anomalies.write
        .format("delta")
        .mode("overwrite")   # snapshot table
        .option("overwriteSchema", "true")
        .save(f"{GOLD_PATH}/anomalies_table")
)

print(f"\nAnomalous records detected: {df_anomalies.count()}")
display(df_anomalies.limit(20))


Anomalous records detected: 347


coin_id,name,symbol,current_price,price_change,total_volume,anomaly_reason,ingestion_time
black-phoenix,Black Phoenix,bpx,0.00405539,3954.9623,3,PRICE_SPIKE,2026-04-22T10:20:59.258Z
pnp-exchange,PNP Exchange,pnp,2.3207E-4,2780.951,23,PRICE_SPIKE,2026-04-22T10:20:59.258Z
e4c,E4C,e4c,0.00109855,1319.0779,109,PRICE_SPIKE,2026-04-22T10:20:59.258Z
alan-the-alien,Alan the Alien,alan,1.142E-5,727.3867,406817,PRICE_SPIKE,2026-04-22T10:20:59.258Z
monitoring-the-situation,Monitoring the Situation,monitor,3.178E-4,544.3051,1580988,PRICE_SPIKE,2026-04-22T10:20:59.258Z
pepi-2,PEPi,pepi,14.71,514.8423,85284,PRICE_SPIKE,2026-04-22T10:20:59.258Z
autobahn-network,Autobahn Network,txl,0.00155694,374.3034,16,PRICE_SPIKE,2026-04-22T10:20:59.258Z
ice,Ice Open Network,ice,6.5827E-4,292.5092,3886,PRICE_SPIKE,2026-04-22T10:20:59.258Z
make-aliens-great-again,Make Aliens Great Again,maga,0.00962799,287.3466,2544923,PRICE_SPIKE,2026-04-22T10:20:59.258Z
hybux,HYBUX,hybux,1.8345E-4,270.667,3961,PRICE_SPIKE,2026-04-22T10:20:59.258Z
